# Cassegrain Antenna Design

This notebook demonstrates the Cassegrain dual-reflector antenna design module in SpaceLink.

The design follows Granet (1998), Table 2, Set No. 1: minimum-blockage condition with
feed position specified via the axial distance **Lm** from the main reflector apex to the
subreflector apex.

**Inputs required:**
- `Dm` — main reflector diameter
- `F_over_D` — focal ratio F/D
- `Lm` — axial distance from main reflector apex to subreflector apex
- `Df` — feed aperture diameter (sets the minimum-blockage condition)
- `frequency` — operating frequency (for gain calculation)
- `efficiency` — overall antenna efficiency η

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

from spacelink.core.cassegrain import (
    cassegrain_gain,
    design_min_blockage,
    main_reflector_profile,
    subreflector_profile,
)

## 1. Design the Geometry

Using the Excel-verified reference case from the design document.

In [ ]:
frequency = 2.2 * u.GHz
efficiency = 0.65 * u.dimensionless_unscaled

geom = design_min_blockage(
    Dm=5 * u.m,
    F_over_D=0.317 * u.dimensionless_unscaled,
    Lm=1.1 * u.m,
    Df=0.1 * u.m,
)

print("=== Cassegrain Geometry ===")
print(f"Main reflector diameter  Dm     = {geom.Dm:.4f}")
print(f"Focal length             F      = {geom.F:.4f}")
print(f"Focal ratio              F/D    = {geom.F / geom.Dm:.4f}")
print(f"Main-to-subrefl distance Lm     = {geom.Lm:.4f}")
print(f"Subreflector diameter    Ds     = {geom.Ds:.9f}")
print(f"Subrefl-to-feed distance Ls     = {geom.Ls:.9f}")
print(f"Semi-transverse axis     a      = {geom.a:.9f}")
print(f"Focus half-distance      f      = {geom.f:.4f}")
print(f"Edge half-angle          theta_e= {geom.theta_e:.9f}")
print(f"Eccentricity             e      = {geom.eccentricity:.9f}")
print(f"Total length             Lm+Ls  = {geom.total_length:.9f}")

## 2. Gain

In [ ]:
gain = cassegrain_gain(geom, frequency, efficiency)
print(f"Frequency   : {frequency}")
print(f"Efficiency η: {efficiency}")
print(f"Gain        : {gain:.2f}")

## 3. Cross-Section Diagram

The diagram below shows:
- **Blue** — main paraboloid and subreflector profiles
- **Green dashed** — incoming parallel rays and reflection off the main dish toward the paraboloid focus
- **Orange dotted** — rays from the subreflector to the feed phase centre
- **Teal marker** — feed phase centre (far focus of the hyperboloid, z = −2f)
- **Grey dot** — paraboloid focus / near focus of the hyperboloid (z = 0)

The feed sits between the main dish vertex and the subreflector at z = −2f,
which is the far focus of the hyperboloid subreflector.

In [ ]:
def plot_cassegrain(geom, n_rays=9, figsize=(9, 11)):
    """Plot a cross-section of a Cassegrain antenna with ray tracing.

    Parameters
    ----------
    geom : CassegrainGeometry
        Geometry produced by design_min_blockage.
    n_rays : int
        Number of incoming rays to draw (above the axis; mirrored below).
    figsize : tuple
        Matplotlib figure size.

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
    """
    # --- Extract scalar values in metres ---
    Dm = geom.Dm.to(u.m).value
    F = geom.F.to(u.m).value
    Ds = geom.Ds.to(u.m).value
    a = geom.a.to(u.m).value
    f = geom.f.to(u.m).value
    b2 = f**2 - a**2  # f² - a²

    # Feed is at the far focus of the hyperboloid: z = -2f.
    # This places it between the main dish vertex (z = -F) and the subreflector
    # apex (z = a - f), at a distance Ls = a + f from the subreflector apex.
    z_feed = -2 * f

    # --- Profile functions (scalar/array, values in metres) ---
    def mr(r):
        """Main reflector z-coordinate at radial distance r."""
        return r**2 / (4 * F) - F

    def sr(r):
        """Subreflector z-coordinate at radial distance r."""
        return a * np.sqrt(1 + r**2 / b2) - f

    # --- Figure ---
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect("equal")
    ax.set_facecolor("white")

    # z-axis line
    ax.axhline(0, color="silver", lw=0.8, zorder=0)

    # Main reflector (top and bottom)
    r_dish = np.linspace(0, Dm / 2, 400)
    z_dish = mr(r_dish)
    ax.plot(z_dish, r_dish, color="#2c3e8c", lw=3, solid_capstyle="round", zorder=3)
    ax.plot(z_dish, -r_dish, color="#2c3e8c", lw=3, solid_capstyle="round", zorder=3)

    # Subreflector (top and bottom)
    r_sub = np.linspace(0, Ds / 2, 200)
    z_sub_curve = sr(r_sub)
    ax.plot(z_sub_curve, r_sub, color="#2c3e8c", lw=3, solid_capstyle="round", zorder=3)
    ax.plot(
        z_sub_curve, -r_sub, color="#2c3e8c", lw=3, solid_capstyle="round", zorder=3
    )

    # Feed phase centre (far focus of hyperboloid, between dish and subreflector)
    ax.plot(
        z_feed,
        0,
        ">",
        color="#1a9e8c",
        ms=10,
        zorder=5,
        label=f"Feed phase centre  (z = {z_feed:.3f} m)",
    )

    # Paraboloid focus / near focus of hyperboloid
    ax.plot(
        0, 0, "o", color="#aaaaaa", ms=6, zorder=4, label="Paraboloid focus  (z = 0)"
    )

    # --- Ray tracing ---
    # Incoming rays span from just outside the subreflector shadow to the dish edge.
    r_rays = np.linspace(Ds / 2 * 1.05, Dm / 2, n_rays)
    # Rays arrive from the right; start them at a comfortable positive z.
    z_ray_start = 0.3 * Dm

    for r_in in r_rays:
        z_mr_hit = mr(r_in)

        # Find where the reflected ray (directed toward paraboloid focus z=0) hits
        # the subreflector.  Parameterise as z(u) = (u/r_in)*z_mr_hit, y(u) = u.
        # Solve: (u/r_in)*z_mr_hit == sr(u)
        c = z_mr_hit / r_in  # negative constant (z_mr_hit < 0)

        def residual(u, _c=c):
            return _c * u - sr(u)

        try:
            fa = residual(1e-12)
            fb = residual(Ds / 2)
            if fa * fb > 0:
                continue
            u_hit = brentq(residual, 1e-12, Ds / 2, xtol=1e-10)
        except Exception:
            continue

        z_sr_hit = sr(u_hit)

        # Draw above the axis
        ax.plot(
            [z_ray_start, z_mr_hit],
            [r_in, r_in],
            "--",
            color="#3a9e3a",
            lw=1.1,
            zorder=2,
        )
        ax.plot(
            [z_mr_hit, z_sr_hit], [r_in, u_hit], "--", color="#3a9e3a", lw=1.1, zorder=2
        )
        ax.plot([z_sr_hit, z_feed], [u_hit, 0], ":", color="#e07020", lw=1.5, zorder=2)

        # Mirror below the axis
        ax.plot(
            [z_ray_start, z_mr_hit],
            [-r_in, -r_in],
            "--",
            color="#3a9e3a",
            lw=1.1,
            zorder=2,
        )
        ax.plot(
            [z_mr_hit, z_sr_hit],
            [-r_in, -u_hit],
            "--",
            color="#3a9e3a",
            lw=1.1,
            zorder=2,
        )
        ax.plot([z_sr_hit, z_feed], [-u_hit, 0], ":", color="#e07020", lw=1.5, zorder=2)

    # --- Labels and formatting ---
    ax.set_xlabel("z (m)", fontsize=12)
    ax.set_ylabel("r (m)", fontsize=12)
    ax.set_title(
        f"Cassegrain Antenna Cross-Section\n"
        f"Dm = {Dm:.2f} m,  F/D = {F / Dm:.3f},  Ds = {Ds:.4f} m",
        fontsize=13,
    )

    info = (
        f"Dm = {Dm:.3f} m\n"
        f"F/D = {F / Dm:.3f}\n"
        f"Ds = {Ds:.4f} m\n"
        f"e = {geom.eccentricity:.4f}\n"
        f"Lm = {geom.Lm.to(u.m).value:.4f} m\n"
        f"Ls = {geom.Ls.to(u.m).value:.4f} m"
    )
    ax.text(
        0.02,
        0.04,
        info,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment="bottom",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8),
    )

    ax.legend(loc="upper right", fontsize=9)
    ax.set_xlim(-F - 0.15 * Dm, z_ray_start + 0.05 * Dm)
    ax.set_ylim(-Dm / 2 - 0.1 * Dm, Dm / 2 + 0.1 * Dm)
    fig.tight_layout()
    return fig, ax


fig, ax = plot_cassegrain(geom)
plt.show()

## 4. Reflector Surface Profiles

Plot the axial depth of both reflectors as a function of radial distance from the axis.

In [ ]:
r_dish = np.linspace(0, geom.Dm / 2, 300)
z_mr = main_reflector_profile(r_dish, geom.F)

r_subrefl = np.linspace(0 * u.m, geom.Ds / 2, 200)
z_sr = subreflector_profile(r_subrefl, geom.a, geom.f)

fig2, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(r_dish.to(u.m).value, z_mr.to(u.m).value, color="#2c3e8c")
axes[0].set_xlabel("r (m)")
axes[0].set_ylabel("z (m)")
axes[0].set_title("Main Reflector Profile (paraboloid)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(r_subrefl.to(u.m).value, z_sr.to(u.m).value, color="#2c3e8c")
axes[1].set_xlabel("r (m)")
axes[1].set_ylabel("z (m)")
axes[1].set_title("Subreflector Profile (hyperboloid)")
axes[1].grid(True, alpha=0.3)

fig2.tight_layout()
plt.show()